# Original EDA Notebook — Cleaned

**Project:** NYC Taxi Trip Duration & Late-Risk Prediction  
**Purpose:** Explore airport taxi trips, pickup patterns, airport differences, and late-risk patterns.

> This notebook is a cleaned portfolio version of the original modeling work. Outputs were cleared for GitHub readability. The project-level README contains the final summarized results and interpretation.


## Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 200)
sns.set_theme()



## Load and Explore the Taxi and Zone Data

In [ ]:
# Load Data
taxi = pd.read_csv('taxi_data.csv')
zones = pd.read_csv('taxi_zone_lookup.csv')

display(taxi.head(3))
display(zones.head(3))

#### Explore Data Types

In [ ]:
taxi.info()
print("--------------------------------")
zones.info()

### Explore Missing Values

In [ ]:
# Missing values for taxi data
display(taxi.isnull().sum())

# Missing values for zones data
display(zones.isnull().sum())

### Convert the Date and Time to timesereies data. Also convert trip duration to minutes and create new column to explore

In [ ]:
# Convert pickup and dropoff to datetime
taxi["tpep_pickup_datetime"] = pd.to_datetime(taxi["tpep_pickup_datetime"])
taxi["tpep_dropoff_datetime"] = pd.to_datetime(taxi["tpep_dropoff_datetime"])

# Trip duration in minutes
taxi["duration_min"] = ((taxi["tpep_dropoff_datetime"] - taxi["tpep_pickup_datetime"]).dt.total_seconds() / 60).round(2)
taxi[["tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_min"]].head()

### Identify the Airport Zones and the Manhattan Zones we are working with

In [ ]:
# Look at airport zones
airports_lookup = zones[zones["Zone"].str.contains("Airport", case=False, na=False)]
print(airports_lookup)

# Get the IDs of the airports
JFK_ID = 132
LGA_ID = 138 
airport_ids = [JFK_ID, LGA_ID]   

# All Manhattan pickup zones with LocationID and Zone
manhattan_zones = zones.loc[zones["Borough"] == "Manhattan", ["LocationID","Zone"]].copy()
manhattan_ids = manhattan_zones["LocationID"].tolist()

# View the zones and IDs
manhattan_zones.head()
len(manhattan_zones)

### Outlier Detection and Removal

##### Restricted the data to trips from Manhattan to JFK or LaGuardia, which gave us about 50,297 rides. Trip durations ranged from 0 minutes to roughly 1,438 minutes (24 hoursish), which is clearly unrealistic for an airport trip. The raw histogram shows a heavy right tail driven by these extreme values.

In [ ]:
# Manhattan to JFK/LGA only
mask = taxi["PULocationID"].isin(manhattan_ids) & taxi["DOLocationID"].isin(airport_ids)

taxi_ma = taxi[mask]
print("Manhattan to JFK/LGA trips (before outlier removal):", taxi_ma.shape)

# Plot duration distribution to spot outliers, zooming in to more relevant duration range
plt.figure(figsize=(8,5))
plt.hist(taxi_ma["duration_min"], bins=50)
plt.title("Trip Duration: Manhattan → JFK/LGA (before outlier removal)")
plt.xlabel("Duration (min)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Outlier mask
mask_outliers = (taxi_ma["duration_min"] <= 3) | (taxi_ma["duration_min"] >= 180)
print("Trips with duration <=3 or >=180:", mask_outliers.sum())

# Drop outliers into a new DF to see what's going on
taxi_clean = taxi_ma.loc[~mask_outliers].copy()
print("After outlier removal:", taxi_clean.shape)

# Replace the old taxi DF with the cleaned 
taxi = taxi_clean

# Plot histogram after outlier removal
plt.figure(figsize=(8,5))
plt.hist(taxi["duration_min"], bins=50)
plt.title("Trip Duration: Manhattan → JFK/LGA (after outlier removal)")
plt.xlabel("Duration (min)")
plt.ylabel("Count")
plt.show()


##### To remove likely trip errors and non standard rides (multi stop, etc.) I treated trips shorter than 3 minutes or longer than 180 minutes as outliers. This affected 696 trips (only about 1.4% of the data), leaving us with 49,601 rides for modeling. The cleaned distribution still has a right tail, but now reflects plausible airport travel times.

In [ ]:
# Add pickup zone name to the cleaned taxi data
taxi = taxi.merge(
    manhattan_zones.rename(columns={"LocationID": "PULocationID", "Zone": "pickup_zone"}),
    on="PULocationID",
    how="left"
)

# Quick check
taxi[["PULocationID", "pickup_zone", "DOLocationID", "duration_min"]].head()


### Time Features and Airport Labels

In [ ]:
# Hour of day and day of week
taxi["hour"] = taxi["tpep_pickup_datetime"].dt.hour
taxi["day_of_week"] = taxi["tpep_pickup_datetime"].dt.dayofweek

# Map airport ID to readable name
airport_map = {JFK_ID: "JFK", LGA_ID: "LGA"}
taxi["airport"] = taxi["DOLocationID"].map(airport_map)

# Quick check
taxi[["PULocationID", "pickup_zone", "DOLocationID", "airport", "hour", "day_of_week", "duration_min"]].head()


In [ ]:
# Baseline which is the median duration by pickup zone, airport, hour.
baseline = (taxi.groupby(["PULocationID", "airport", "hour"])["duration_min"].median().rename("median_duration").reset_index())
baseline.head()

# Merge baseline back into main df
taxi = taxi.merge(
    baseline,
    on=["PULocationID", "airport", "hour"],
    how="left")

# Defining late as more than 20% longer than median for that zone+airport+hour
taxi["late"] = (taxi["duration_min"] > 1.2 * taxi["median_duration"]).astype(int)
taxi.head()

### Late Class Balance

In [ ]:
# Check balance of late class
taxi["late"].value_counts(normalize=True)
taxi.groupby("airport")["late"].value_counts(normalize=True)

### Distribution of Trip Duration

In [ ]:
# Distribution of Trip Duration
plt.hist(taxi["duration_min"], bins=50)
plt.title("Distribution of Trip Duration (minutes)")
plt.xlabel("Duration (min)")
plt.ylabel("Count")
plt.show()

### Trip Duration by Airport

In [ ]:
plt.figure(figsize=(6, 5))
taxi.boxplot(column="duration_min", by="airport")
plt.title("Trip Duration by Airport")
plt.suptitle("")  
plt.xlabel("Airport")
plt.ylabel("Duration (min)")
plt.show()

### Average Trip Duration by Hour and Airport

In [ ]:
# Average duration by pickup hour and airport
avg_by_hour = (taxi.groupby(["hour", "airport"])["duration_min"].mean().reset_index().sort_values("hour"))

fig, ax = plt.subplots(figsize=(8, 6))

# One line per airport
for airport, grp in avg_by_hour.groupby("airport"):
    ax.plot(grp["hour"], grp["duration_min"], marker="o", label=airport)

# Plot 
ax.set_title("Average Trip Duration by Hour and Airport")
ax.set_xlabel("Pickup Hour")
ax.set_ylabel("Average Duration (min)")
ax.set_xticks(range(0, 24))   # 0–23
ax.set_xlim(0, 23)
ax.legend(title="Airport")
plt.show()


### Average Trip Duration by Day of Week 

In [ ]:
avg_by_day = (taxi.groupby("day_of_week")["duration_min"].mean().reset_index())

plt.figure(figsize=(8, 6))
plt.plot(avg_by_day["day_of_week"], avg_by_day["duration_min"], marker="o")
plt.title("Average Trip Duration by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Duration (min)")
plt.xticks(range(0, 7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.show()


# Average trip duration by day of week and airport
avg_by_day_airport = (taxi.groupby(["day_of_week", "airport"])["duration_min"].mean().reset_index())

plt.figure(figsize=(8, 6))
for airport, grp in avg_by_day_airport.groupby("airport"):
    plt.plot(grp["day_of_week"], grp["duration_min"], marker="o", label=airport)
plt.title("Average Trip Duration by Day of Week and Airport")
plt.xlabel("Day of Week")
plt.ylabel("Average Duration (min)")
plt.xticks(range(0, 7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.legend(title="Airport")
plt.show()


### Late Rate by Hour (All Airports)

In [ ]:
# Late Rate by Hour
late_by_hour = (taxi.groupby("hour")["late"].mean().reset_index().rename(columns={"late": "late_rate"}))

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(late_by_hour["hour"], late_by_hour["late_rate"], marker="o")
ax.set_title("Late Rate by Pickup Hour (All Airports)")
ax.set_xlabel("Pickup Hour")
ax.set_ylabel("Late Rate")
ax.set_xticks(range(0, 24))
ax.set_ylim(0, 1)
plt.show()

## Late Rate by Hour AND Airport

In [ ]:
# Late Rate by Hour and Airport
late_by_airport_hour = (taxi.groupby(["airport", "hour"])["late"].mean().reset_index().rename(columns={"late": "late_rate"}))

for a in late_by_airport_hour["airport"].unique():
    subset = late_by_airport_hour[late_by_airport_hour["airport"] == a]
    plt.plot(subset["hour"], subset["late_rate"], marker="o", label=a)

# PLot
plt.title("Late Rate by Hour and Airport")
plt.xlabel("Pickup Hour")
plt.ylabel("Late Rate")
plt.xticks(range(0, 24))
plt.ylim(0, 1)
plt.legend()
plt.show()


### Late Rate by Day of Week

In [ ]:
late_by_dow = (
    taxi.groupby("day_of_week")["late"]
    .mean()
    .reset_index()
)
plt.plot(late_by_dow["day_of_week"], late_by_dow["late"], marker="o")
plt.xticks(range(7), ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
plt.title("Late Rate by Day of Week")
plt.ylabel("Late Rate")
plt.show()


In [ ]:
late_by_dow_airport = (
    taxi.groupby(["day_of_week", "airport"])["late"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
for airport in late_by_dow_airport["airport"].unique():
    plt.plot(
        late_by_dow_airport[late_by_dow_airport["airport"] == airport]["day_of_week"],
        late_by_dow_airport[late_by_dow_airport["airport"] == airport]["late"],
        marker="o",
        label=airport
    )
plt.xticks(range(7), ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
plt.title("Late Rate by Day of Week and Airport")
plt.ylabel("Late Rate")
plt.xlabel("Day of Week")
plt.legend(title="Airport")
plt.show()


# Percentage late by day of week, grouped bar by airport
late_by_dow_airport = (taxi.groupby(["day_of_week", "airport"])["late"].mean().reset_index())

plt.figure(figsize=(8, 5))
width = 0.35
airports = late_by_dow_airport["airport"].unique()
days = list(range(7))
x = np.arange(len(days))

for i, airport in enumerate(airports):
    y = late_by_dow_airport[late_by_dow_airport["airport"] == airport].set_index("day_of_week")["late"].reindex(days, fill_value=0).values
    plt.bar(x + i*width, y, width=width, label=airport)

plt.xticks(x + width/2, ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
plt.title("Proportion of Late Trips by Day of Week and Airport")
plt.ylabel("Proportion Late")
plt.xlabel("Day of Week")
plt.legend(title="Airport")
plt.tight_layout()
plt.show()



### Zone EDA

In [ ]:
# Duration by Pickup Zone 
plt.figure(figsize=(12, 6))
duration_by_zone_airport = (taxi.groupby(['pickup_zone', 'airport'])['duration_min'].median().reset_index())

# Plot
for airport in duration_by_zone_airport['airport'].unique():
    data = duration_by_zone_airport[duration_by_zone_airport['airport'] == airport]
    plt.bar(data['pickup_zone'], data['duration_min'], label=airport, alpha=0.7)
plt.xlabel('Pickup Zone')
plt.ylabel('Median Duration (min)')
plt.title('Median Trip Duration by Pickup Zone to Each Airport')
plt.legend(title="Airport")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

#  Top Pickup Zones by Median Duration (to JFK and LGA)
for airport in ['JFK', 'LGA']:
    top_durations = duration_by_zone_airport[duration_by_zone_airport['airport']==airport]
    top_durations = top_durations.sort_values('duration_min', ascending=False).head(15)
    plt.figure(figsize=(8, 5))
    plt.barh(top_durations['pickup_zone'], top_durations['duration_min'], color='teal')
    plt.xlabel('Median Duration (minutes)')
    plt.title(f'Top 15 Pickup Zones by Median Duration to {airport}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# Late Rate by Pickup Zone to each airport
late_by_zone_airport = (taxi.groupby(['pickup_zone', 'airport'])['late'].mean().reset_index())
for airport in late_by_zone_airport['airport'].unique():
    zone_late = late_by_zone_airport[late_by_zone_airport['airport']==airport]
    top_late = zone_late.sort_values('late', ascending=False).head(15)
    plt.figure(figsize=(9, 5))
    plt.barh(top_late['pickup_zone'], top_late['late'], color='orange')
    plt.xlabel('Proportion Late')
    plt.title(f'Top 15 Pickup Zones by Highest Late Rate to {airport}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# Duration Distribution by Pickup Zone to JFK and LGA
import seaborn as sns

for airport in ['JFK', 'LGA']:
    zone_counts = taxi[taxi['airport']==airport]['pickup_zone'].value_counts()
    top_zones = zone_counts.head(10).index.tolist()
    plt.figure(figsize=(12, 6))
    sns.boxplot(
        x='pickup_zone', y='duration_min',
        data=taxi[(taxi['airport']==airport) & (taxi['pickup_zone'].isin(top_zones))],
        showfliers=False
    )
    plt.title(f'Trip Duration Distribution by Top 10 Pickup Zones to {airport}')
    plt.xlabel('Pickup Zone')
    plt.ylabel('Duration (min)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

#  Zone EDA Summary Table 
zone_summary = taxi.groupby(['pickup_zone', 'airport']).agg(
    n_trips = ('duration_min', 'count'),
    median_duration = ('duration_min', 'median'),
    late_rate = ('late', 'mean')
).reset_index()
display(zone_summary.sort_values('n_trips', ascending=False).head(15))

In [ ]:
taxi.to_csv("taxi_clean_for_modeling.csv", index=False)
print("Saved taxi_clean_for_modeling.csv")
